<a href="https://colab.research.google.com/github/dasosis/soc-soh-prediction/blob/main/battery_bench.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%cd /content/drive/MyDrive
!git clone https://github.com/dasosis/soc-soh-prediction.git battery-bench   # one-time
%cd /content/drive/MyDrive/battery-bench
!git pull

/content/drive/MyDrive
fatal: destination path 'battery-bench' already exists and is not an empty directory.
/content/drive/MyDrive/battery-bench
remote: Enumerating objects: 30, done.
remote: Counting objects: 100% (30/30), done.
remote: Compressing objects: 100% (10/10), done.
remote: Total 20 (delta 13), reused 16 (delta 9), pack-reused 0 (from 0)
Unpacking objects: 100% (20/20), 15.23 KiB | 7.00 KiB/s, done.
From https://github.com/dasosis/soc-soh-prediction
   a6f889e..bbd6b39  main       -> origin/main
Updating 4a219d7..bbd6b39
Fast-forward
 README.md                         |  28 +++
 THESIS_LOG.md                     |  15 ++
 configs/soc_stage2.yaml           |  69 +++++++
 scripts/run_soc_stage1.py         |  25 ++-
 scripts/run_soc_stage2.py         | 417 ++++++++++++++++++++++++++++++++++++++
 src/battery_bench/soc_finetune.py |  89 ++++++++
 tests/test_models_stage1.py       |  27 +++
 tests/test_stage2_finetune.py     |  66 ++++++
 8 files changed, 732 insertions(+), 4 de

In [3]:
!pip install -e . -q
import torch; print(torch.__version__, "CUDA:", torch.cuda.is_available())

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for battery-bench (pyproject.toml) ... done
2.11.0+cu128 CUDA: True


In [ ]:
#pip install --force-reinstall torch

In [4]:
%cd /content/drive/MyDrive/battery-bench
!unzip -o soc_crossdataset_bundle.zip -d .
!ls data/processed/ && ls runs/soc_cross_dataset/

/content/drive/MyDrive/battery-bench
Archive:  soc_crossdataset_bundle.zip
  inflating: ./processed/cells.parquet  
  inflating: ./processed/soc_timeseries.parquet  
  inflating: ./SCHEMA.md             
  inflating: ./runs/soc_cross_dataset/fold_A_test_LG_HG2.json  
  inflating: ./runs/soc_cross_dataset/fold_B_test_PANASONIC_18650PF.json  
cells.parquet  soc_timeseries.parquet
fold_A_test_LG_HG2.json  fold_B_test_PANASONIC_18650PF.json


In [ ]:
!python scripts/run_soc_stage1.py --config configs/soc_stage1.yaml --smoke

SMOKE sweep | experiments=['within_LG', 'within_PAN', 'cross_A_train_PAN_test_LG', 'cross_B_train_LG_test_PAN'] | models=['linear', 'mlp', 'lstm', 'gru', 'bilstm', 'cnn_bilstm_attn', 'tcn', 'transformer', 'patchtst'] | seeds=[0]

=== within_LG (within) | windows train/val/test = 1500/800/800 | prep 0.8s ===
  linear           seed 0 | MAE 0.0562 RMSE 0.0768 R2 0.898 | 1.2s | 301 params
  mlp              seed 0 | MAE 0.0933 RMSE 0.1192 R2 0.755 | 7.0s | 110,081 params
  lstm             seed 0 | MAE 0.2358 RMSE 0.2692 R2 -0.248 | 0.3s | 38,881 params
  gru              seed 0 | MAE 0.2946 RMSE 0.3183 R2 -0.746 | 0.1s | 29,185 params
  bilstm           seed 0 | MAE 0.4091 RMSE 0.4539 R2 -2.550 | 0.1s | 77,761 params
  cnn_bilstm_attn  seed 0 | MAE 0.2246 RMSE 0.2637 R2 -0.198 | 0.7s | 42,402 params
  tcn              seed 0 | MAE 0.3156 RMSE 0.3707 R2 -1.368 | 0.3s | 16,001 params
  transformer      seed 0 | MAE 0.1564 RMSE 0.1781 R2 0.453 | 0.4s | 67,265 params
  patchtst         seed 

In [ ]:
!python scripts/run_soc_stage1.py --config configs/soc_stage1.yaml

FULL sweep | experiments=['within_LG', 'within_PAN', 'cross_A_train_PAN_test_LG', 'cross_B_train_LG_test_PAN'] | models=['linear', 'mlp', 'lstm', 'gru', 'bilstm', 'cnn_bilstm_attn', 'tcn', 'transformer', 'patchtst'] | seeds=[0, 1, 2, 3, 4]

=== within_LG (within) | windows train/val/test = 31267/6898/6117 | prep 0.8s ===
  skip (done) within_LG/linear/seed_0
  linear           seed 1 | MAE 0.0492 RMSE 0.0691 R2 0.920 | 0.9s | 301 params
  linear           seed 2 | MAE 0.0492 RMSE 0.0691 R2 0.920 | 0.2s | 301 params
  linear           seed 3 | MAE 0.0492 RMSE 0.0691 R2 0.920 | 0.3s | 301 params
  linear           seed 4 | MAE 0.0492 RMSE 0.0691 R2 0.920 | 0.3s | 301 params
  skip (done) within_LG/mlp/seed_0
  mlp              seed 1 | MAE 0.0130 RMSE 0.0176 R2 0.995 | 10.9s | 110,081 params
  mlp              seed 2 | MAE 0.0125 RMSE 0.0164 R2 0.995 | 9.4s | 110,081 params
  mlp              seed 3 | MAE 0.0135 RMSE 0.0173 R2 0.995 | 8.6s | 110,081 params
  mlp              seed 4 | MAE

In [3]:
# 1. Remove the contaminated seed_0 dirs (the smoke run) + stale aggregates
%cd /content/drive/MyDrive/battery-bench
!find runs/soc_stage1 -type d -name seed_0 -exec rm -rf {} +
!rm -rf runs/soc_stage1/diagnostics runs/soc_stage1/summary.csv
!ls runs/soc_stage1/within_LG/lstm/    # confirm: seed_1..4 remain, seed_0 gone

/content/drive/MyDrive/battery-bench
seed_1	seed_2	seed_3	seed_4


In [7]:
!ls data/processed/ && ls runs/soc_cross_dataset/

cells.parquet  soc_timeseries.parquet
fold_A_test_LG_HG2.json  fold_B_test_PANASONIC_18650PF.json


In [8]:
# 2. Re-run — resume skips seeds 1-4, retrains only seed_0 at full config (~30-45 min on GPU),
#    then rewrites summary.csv + diagnostics over a clean 5 seeds
!python scripts/run_soc_stage1.py --config configs/soc_stage1.yaml

FULL sweep | experiments=['within_LG', 'within_PAN', 'cross_A_train_PAN_test_LG', 'cross_B_train_LG_test_PAN'] | models=['linear', 'mlp', 'lstm', 'gru', 'bilstm', 'cnn_bilstm_attn', 'tcn', 'transformer', 'patchtst'] | seeds=[0, 1, 2, 3, 4]

=== within_LG (within) | windows train/val/test = 31267/6898/6117 | prep 0.8s ===
  linear           seed 0 | MAE 0.0492 RMSE 0.0691 R2 0.920 | 1.2s | 301 params
  skip (done) within_LG/linear/seed_1
  skip (done) within_LG/linear/seed_2
  skip (done) within_LG/linear/seed_3
  skip (done) within_LG/linear/seed_4
  mlp              seed 0 | MAE 0.0130 RMSE 0.0168 R2 0.995 | 15.6s | 110,081 params
  skip (done) within_LG/mlp/seed_1
  skip (done) within_LG/mlp/seed_2
  skip (done) within_LG/mlp/seed_3
  skip (done) within_LG/mlp/seed_4
  lstm             seed 0 | MAE 0.0079 RMSE 0.0108 R2 0.998 | 27.9s | 38,881 params
  skip (done) within_LG/lstm/seed_1
  skip (done) within_LG/lstm/seed_2
  skip (done) within_LG/lstm/seed_3
  skip (done) within_LG/lstm

In [9]:
# 3. Verify the contamination is gone: linear is deterministic, so its seed std MUST be ~0 now
import pandas as pd
df = pd.read_csv('runs/soc_stage1/summary.csv')
print(df[df.model == 'linear'][['experiment','mae_mean','mae_std']])
# mae_std ~0.000 for linear, and the deep-model stds should collapse from ~0.1-0.2 to single-digit %

                   experiment  mae_mean  mae_std
3   cross_A_train_PAN_test_LG   0.08165      0.0
12  cross_B_train_LG_test_PAN   0.07366      0.0
21                  within_LG   0.04918      0.0
30                 within_PAN   0.06858      0.0


In [12]:
#git results
%rm -rf results/stage1
%mkdir -p results/stage1
%cp runs/soc_stage1/summary.csv runs/soc_stage1/results.csv results/stage1/
%cp -r runs/soc_stage1/diagnostics results/stage1/

In [14]:
!git add results/stage1

In [19]:
import os
!git config --global user.name "dasosis"
!git config --global user.email "soumyasnigdhadas@gmail.com"

In [20]:
commit_message = "Add stage1 results"
!git commit -m "{commit_message}"

[main 4a219d7] Add stage1 results
 7 files changed, 644 insertions(+)
 create mode 100644 results/stage1/diagnostics/bias_summary.csv
 create mode 100644 results/stage1/diagnostics/error_by_soc.csv
 create mode 100644 results/stage1/diagnostics/error_by_soc_cross_A_train_PAN_test_LG.png
 create mode 100644 results/stage1/diagnostics/error_by_soc_cross_B_train_LG_test_PAN.png
 create mode 100644 results/stage1/diagnostics/within_vs_cross_gap.csv
 create mode 100644 results/stage1/results.csv
 create mode 100644 results/stage1/summary.csv


In [21]:
from google.colab import userdata
GH_TOKEN = userdata.get('GH_TOKEN')

remote_url = !git remote get-url origin
repo_url = remote_url[0].strip()

if repo_url.startswith('https://github.com/'):
    authenticated_repo_url = repo_url.replace('https://github.com/', f'https://{GH_TOKEN}@github.com/')
    print(f"Pushing to {authenticated_repo_url.replace(GH_TOKEN, '********')}")
    !git push "{authenticated_repo_url}"
else:
    print("Could not determine a standard GitHub HTTPS URL for remote 'origin'. Please push manually or specify the full authenticated URL.")

Pushing to https://********@github.com/dasosis/soc-soh-prediction.git
Enumerating objects: 13, done.
Counting objects: 100% (13/13), done.
Delta compression using up to 2 threads
Compressing objects: 100% (11/11), done.
Writing objects: 100% (12/12), 420.45 KiB | 8.08 MiB/s, done.
Total 12 (delta 1), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (1/1), completed with 1 local object.
To https://github.com/dasosis/soc-soh-prediction.git
   a6f889e..4a219d7  main -> main


In [5]:
!python scripts/run_soc_stage2.py --config configs/soc_stage2.yaml

FULL stage-2 | directions=['cross_A_train_PAN_test_LG', 'cross_B_train_LG_test_PAN'] | models=['bilstm', 'cnn_bilstm_attn', 'gru', 'linear', 'lstm', 'mlp', 'patchtst', 'tcn', 'transformer'] | fractions=[0.0, 0.05, 0.1, 0.2, 0.5, 1.0] | seeds=[0, 1, 2, 3, 4]

=== cross_A_train_PAN_test_LG | source_tr/val=42051/10582 test/val/pool windows=17177/7399/19706 | profiles test/val/pool=27/12/30 | prep 2.2s ===
  bilstm         seed 0 | src 77.3s | methods=['recal_affine', 'recal_bias'] | zero-shot MAE 0.0870
  bilstm         seed 1 | src 41.5s | methods=['recal_affine', 'recal_bias'] | zero-shot MAE 0.0853
  bilstm         seed 2 | src 88.5s | methods=['recal_affine', 'recal_bias'] | zero-shot MAE 0.0828
  bilstm         seed 3 | src 46.4s | methods=['recal_affine', 'recal_bias'] | zero-shot MAE 0.0805
  bilstm         seed 4 | src 41.0s | methods=['recal_affine', 'recal_bias'] | zero-shot MAE 0.0798
  cnn_bilstm_attn seed 0 | src 27.2s | methods=['recal_affine', 'recal_bias'] | zero-shot MAE 